In [3]:
import pandas as pd
import numpy as np

In [4]:
data = pd.read_csv('data/GSE115469_Data.csv', index_col=0)

In [5]:
cell_cluster_data = pd.read_csv('data/GSE115469_CellClusterType.txt', sep='\t')

In [6]:
data.shape

(20007, 8444)

In [7]:
data.head()

,P1TLH_AAACCTGAGCAGCCTC_1,P1TLH_AAACCTGTCCTCATTA_1,P1TLH_AAACCTGTCTAAGCCA_1,P1TLH_AAACGGGAGTAGGCCA_1,P1TLH_AAACGGGGTTCGGGCT_1,P1TLH_AAAGCAACAGTAAGAT_1,P1TLH_AAAGCAAGTCGCGTGT_1,P1TLH_AAAGCAAGTGTTTGTG_1,P1TLH_AAAGCAAGTTGATTCG_1,P1TLH_AAAGTAGCAGACGTAG_1,...,P5TLH_TTTCCTCTCAGTGTTG_1,P5TLH_TTTGCGCAGGATGGTC_1,P5TLH_TTTGCGCCAATGACCT_1,P5TLH_TTTGCGCCATCCTAGA_1,P5TLH_TTTGTCAGTCAGGACA_1,P5TLH_TTTGTCAGTGTTCTTT_1,P5TLH_TTTGTCAGTTTAGGAA_1,P5TLH_TTTGTCATCAGCTTAG_1,P5TLH_TTTGTCATCCACGCAG_1,P5TLH_TTTGTCATCGGCATCG_1
RP11-34P13.7,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
FO538757.2,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
AP006222.2,0.0,0.31476,0.0,0.0,0.0,0.0,0.0,0.504068,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.140429,0.0,0.0,0.0
RP4-669L17.10,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
RP5-857K21.4,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [6]:
data.isna().values.any()

np.False_

# PRETPROCESIRANJE - Opšti koraci

## Korak 1: Uklanjanje neinformativnih gena
U početnoj fazi pretprocesiranja izvršeno je uklanjanje gena koji ne doprinose značajno klasifikaciji ćelijskih tipova. Single-cell RNA-seq podaci karakterišu se velikim brojem gena, pri čemu se veliki deo njih ne eksprimuje ili pokazuje veoma malu varijabilnost između različitih ćelija. Takvi geni mogu povećati dimenzionalnost skupa podataka, uneti šum i negativno uticati na performanse klasifikacionih modela.


### 1.1 Geni koji se eksprimuju u manje od 5% ćelija

Prvi korak predstavlja uklanjanje gena koji se pojavljuju u malom broju ćelija. Za svaki gen izračunata je učestalost njegove ekspresije, odnosno procenat ćelija u kojima je nivo ekspresije veći od nule. Zadržani su samo geni koji su eksprimirani u najmanje 5% ćelija.

In [ ]:
gene_frequency = (data > 0).mean(axis=1)
data = data.loc[gene_frequency >= 0.05]

In [ ]:
data.shape

(6717, 8444)

### 1.2 Geni koji imaju skoro isti nivo ekspresije u svakoj ćeliji
Uklanjaju se geni sa veoma sličnim nivoom ekspresije kroz sve ćelije. Za svaki gen izračunata je varijansa ekspresije. Geni su zatim sortirani prema vrednosti varijanse i izabrano je 2000 gena sa najvećom varijabilnošću. Geni sa većom varijansom predstavljaju gene čiji se nivoi ekspresije razlikuju između ćelija, pa samim tim nose više informacija za razlikovanje različitih ćelijskih tipova. Zadržavanjem najvarijabilnijih gena smanjuje se broj atributa, ubrzava treniranje modela i poboljšava sposobnost algoritama da pronađu relevantne obrasce u podacima.

In [9]:
gene_var = data.var(axis=1)
gene_var_sorted = gene_var.sort_values(ascending=False)
gene_var_keep = gene_var_sorted[:2000]

In [10]:
gene_var_keep

APOC3     12.340822
APOA2     11.126836
ORM1       9.849425
HP         9.393121
ALB        9.389534
            ...    
TACC1      0.375015
PDXDC1     0.374714
FUNDC2     0.374669
RNF167     0.374545
PSAT1      0.374325
Length: 2000, dtype: float64

In [11]:
data = data.loc[gene_var_keep.index]

In [12]:
data.shape

(2000, 8444)

Nakon ovog koraka broj atributa je smanjen sa početnog broja gena na 2000 najinformativnijih gena, čime je izvršena redukcija dimenzionalnosti uz očuvanje najvažnijih karakteristika genske ekspresije.

## Korak 2: Outlier clipping
Nakon selekcije informativnih gena izvršeno je ograničavanje ekstremnih vrednosti ekspresije gena (outlier clipping). Single-cell RNA-seq podaci mogu sadržati pojedinačne veoma visoke vrednosti ekspresije koje nastaju usled tehničkih faktora, kao što su razlike u broju očitanih transkripata ili šum tokom merenja.

Kako bi se smanjio uticaj takvih ekstremnih vrednosti na proces klasifikacije, primenjen je clipping zasnovan na 99.9-tom percentilu. Vrednosti ekspresije koje prelaze definisani prag zamenjene su maksimalnom dozvoljenom vrednošću.

In [13]:
data = data.clip(upper=data.quantile(0.999).max())

Ovim postupkom očuvane su informacije sadržane u distribuciji genske ekspresije, dok je istovremeno smanjen uticaj ekstremnih merenja koja bi mogla dominirati prilikom treniranja modela.

## Korak 3: Transponovanje matrice
U originalnoj matrici genske ekspresije podaci su bili organizovani tako da su geni predstavljali redove, a pojedinačne ćelije kolone. Kako klasifikacioni algoritmi zahtevaju format u kome svaki red predstavlja jedan uzorak, a svaka kolona jedan atribut, izvršeno je transponovanje matrice.

Nakon transponovanja:

* svaki red odgovara jednoj ćeliji,  
* svaka kolona predstavlja ekspresiju jednog gena,  
* vrednosti u matrici predstavljaju nivo ekspresije određenog gena u određenoj ćeliji.

Dobijena matrica predstavlja ulazni skup atributa (feature matrix) koji se dalje koristi u procesu klasifikacije ćelijskih tipova.

In [14]:
data = data.T

In [15]:
data.shape

(8444, 2000)